### **Intall Dependencies**

In [ ]:
!pip install -q transformers accelerate datasets torch pandas

# **Confirm GPU and RAM**
You can change the Google Collab runtime:


*   -> Runtime
*   -> Change runtime type
* -> T4 GPU if on free version, A100 if pro




In [ ]:
# Confirm GPU and RAM
!nvidia-smi
!torch.cuda.get_device_name(0)

Mon Dec  1 21:52:26 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# **Load data from google drive folder**

In [ ]:
from google.colab import drive, files
import pandas as pd
drive.mount('/content/drive')
df = pd.read_csv("mines.csv")
df

Mounted at /content/drive


,Context,Target
0,"Voltage: 0.338156758 V, Height: 0 cm, Soil Typ...",Mine Type: NA
1,"Voltage: 0.320241334 V, Height: 0.181818182 cm...",Mine Type: NA
2,"Voltage: 0.28700875 V, Height: 0.272727273 cm,...",Mine Type: NA
3,"Voltage: 0.256283622 V, Height: 0.454545455 cm...",Mine Type: NA
4,"Voltage: 0.262839599 V, Height: 0.545454545 cm...",Mine Type: NA
...,...,...
333,"Voltage: 0.323262478 V, Height: 0.909090909 cm...",Mine Type: M14 Anti-Personnel
334,"Voltage: 0.444108237 V, Height: 0.181818182 cm...",Mine Type: M14 Anti-Personnel
335,"Voltage: 0.353473918 V, Height: 0.454545455 cm...",Mine Type: M14 Anti-Personnel
336,"Voltage: 0.36253735 V, Height: 0.727272727 cm,...",Mine Type: M14 Anti-Personnel


# **Select precision based on GPU**

In [ ]:
import os
import torch
import pandas as pd
from datetime import datetime
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from datasets import Dataset
import json

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    major, minor = torch.cuda.get_device_capability()
    print(f"GPU detected: {gpu_name} (Compute capability {major}.{minor})")

    # A100 / L4 GPUs support bf16
    if major >= 8:
        torch_dtype = torch.bfloat16
    else:
        torch_dtype = torch.float16
else:
    print("No GPU detected, falling back to CPU.")
    torch_dtype = torch.float32

# Cache model
os.environ["TRANSFORMERS_CACHE"] = "/content/cache"

GPU detected: Tesla T4 (Compute capability 7.5)


# **Run Model**
Outputs are saved to google drive in csv format

In [ ]:
from huggingface_hub import login

login("HF_TOKEN")

# Load model and tokenizer
model_id = "google/gemma-3-4b-it"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    #torch_dtype=torch_dtype,
    dtype=torch.bfloat16,
    device_map="auto"
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto",
    batch_size=8,             # Increase based on GPU capacity
    truncation=True,
    padding=True,
    temperature=0.7,
    max_new_tokens=100,
    return_full_text=False
)


# Load data
df = df.head(8)
dataset = Dataset.from_pandas(df)

# Build all the prompts at once
def build_prompt(batch):
    return {
        "prompt": [
            f"You are a sensor expert. Given the following situation:\n"
            f"Context: {c}\nTarget: Land mine {t}\n\n"
            f"Recommend the most appropriate type of sensor to detect the target and briefly explain why. Be concise."
            for c, t in zip(batch["Context"], batch["Target"])
        ]
    }

dataset = dataset.map(build_prompt, batched=True, num_proc=4)

# Run inference
@torch.inference_mode()
def generate_recommendations(batch):
    outputs = pipe(batch["prompt"])
    return {"sensor_recommendation": [o[0]["generated_text"] for o in outputs]}

dataset = dataset.map(generate_recommendations, batched=True, batch_size=8)

# Save output
df_out = dataset.to_pandas()
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
output_path = f"/content/drive/MyDrive/CMDACapstone/KG{timestamp}.csv"

os.makedirs(os.path.dirname(output_path), exist_ok=True)
df_out.to_csv(output_path, index=False)

print(f"\nSaved to: {output_path}")
print(f"Preview:")
print(df_out.head(3))

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Device set to use cuda:0


Map (num_proc=4):   0%|          | 0/8 [00:00<?, ? examples/s]

In [ ]:
# intercative knowledge graph if they want it
!pip install pyvis
from neo4j import GraphDatabase
from pyvis.network import Network
import re
from collections import Counter
import os

# neo4j
URI = "YOUR_URI"
AUTH = ("USERNAME", "PASSWORD")
driver = GraphDatabase.driver(URI, auth=AUTH)

# extract keywords
stop_words = set([
    "the", "and", "is", "in", "of", "to", "a", "for", "with", "on", "as", "by", "an", "at"
])

def extract_keywords_one_word(text, top_n=1):
    words = re.findall(r'\b[a-zA-Z]{2,}\b', str(text).lower())
    counts = Counter(w for w in words if w not in stop_words)
    return [w for w, _ in counts.most_common(top_n)]

def one_word_label(label):
    if not label:
        return "N/A"
    label = os.path.splitext(str(label))[0]
    words = label.replace("_"," ").replace("-"," ").split()
    word = words[0] if words else "N/A"
    safe = ''.join(c if ord(c)<128 else '?' for c in word)
    return safe

# get nodes
with driver.session() as session:
    node_result = session.run("""
        MATCH (n)
        RETURN elementId(n) AS element_id,
               COALESCE(n.fileName, n.text, n.name, n.id) AS label
    """)
    nodes = []
    for record in node_result:
        nodes.append({
            "id": record["element_id"],
            "label": one_word_label(record["label"])
        })

    edge_result = session.run("""
        MATCH (n)-[r]->(m)
        RETURN elementId(n) AS source, elementId(m) AS target, type(r) AS rel_type
    """)
    edges = []
    for record in edge_result:
        edges.append({
            "from": record["source"],
            "to": record["target"],
            "label": record["rel_type"]
        })

# create pyvis network
net = Network(height="800px", width="100%", notebook=True, bgcolor="#ffffff", font_color="black")
net.barnes_hut()  # better for large graphs

# add nodes
for n in nodes:
    net.add_node(n["id"], label=n["label"], title=n["label"])

# add edges
for e in edges:
    net.add_edge(e["from"], e["to"], title=e["label"])

# show the graph
net.show("knowledge_graph.html")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 56.2 MB/s eta 0:00:00


ModuleNotFoundError: No module named 'neo4j'

# **Saving Attention Weights**
The transformers.pipeline() is designed for high level generation, not for low level model internals like attentions or logits. So we can't use them here.



In [ ]:
%pip install neo4j langchain langchain_openai langchain-community python-dotenv --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.4/325.4 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.3/84.3 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 73.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [ ]:
import torch

# Clear PyTorch GPU memory
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

# Optional: print current memory status
print(torch.cuda.memory_summary(device=0))


|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 1            |        cudaMalloc retries: 1         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |  14354 MiB |  14354 MiB | 333077 MiB | 318723 MiB |
|       from large pool |  14349 MiB |  14349 MiB | 324305 MiB | 309955 MiB |
|       from small pool |      4 MiB |      4 MiB |   8771 MiB |   8767 MiB |
|---------------------------------------------------------------------------|
| Active memory         |  14354 MiB |  14354 MiB | 333077 MiB | 318723 MiB |
|       from large pool |  14349 MiB |  14349 MiB | 324305 MiB |

In [ ]:
import os
import networkx as nx
import matplotlib.pyplot as plt
from neo4j import GraphDatabase
import numpy as np

# neo4j connection
URI = "YOUR_URI"
AUTH = ("USERNAME", "PASSWORD")
driver = GraphDatabase.driver(URI, auth=AUTH)

G = nx.DiGraph()

def make_label(properties):
    """
    Convert node properties into readable label text.
    Priority:
        1. text (shortened)
        2. fileName + page_number
        3. type
        4. element ID
    """
    # if node has "text"
    if "text" in properties and properties["text"]:
        t = properties["text"].strip().replace("\n", " ")
        return t[:60] + "..." if len(t) > 60 else t

    # if node has filename + page
    if "fileName" in properties and "page_number" in properties:
        return f"{properties['fileName']} (page {properties['page_number']})"

    # if node has a type
    if "type" in properties:
        return properties["type"]

    # element id
    return properties.get("id", properties.get("element_id", "Unknown Node"))

# query
node_ids = []
edges = []

with driver.session() as session:
    results = session.run("MATCH (n)-[r]->(m) RETURN n, r, m")

    for record in results:
        n = record["n"]
        m = record["m"]

        # Internal unique IDs
        n_id = n.element_id
        m_id = m.element_id

        # Add nodes with human-readable labels
        G.add_node(n_id, label=make_label(n))
        G.add_node(m_id, label=make_label(m))

        # Record edges
        edges.append((n_id, m_id))

# add edges to graph (relationships)
G.add_edges_from(edges)

# build matrix
node_list = sorted(G.nodes())
adj_matrix = nx.to_numpy_array(G, nodelist=node_list)

print("Adjacency matrix shape:", adj_matrix.shape)

# visualize
plt.figure(figsize=(14, 10))

pos = nx.spring_layout(G, seed=42)

node_labels = nx.get_node_attributes(G, "label")

nx.draw(
    G,
    pos,
    labels=node_labels,
    with_labels=True,
    node_color="skyblue",
    node_size=900,
    font_size=7,
    arrowsize=20
)

plt.title("Knowledge Graph with Human-Readable Labels")
plt.show()

In [ ]:
# debugging to be sure it is finding the files
with driver.session() as session:
    records = session.run("MATCH (n) RETURN n LIMIT 5")
    for r in records:
        print(dict(r["n"]))

NameError: name 'driver' is not defined

In [ ]:
# returns the node labels to be sure it is a meaningful value
for node in G.nodes():
    print(G.nodes[node]["label"])

In [ ]:
import os
import torch
import pandas as pd
from datetime import datetime
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import Dataset
import json
import zipfile
from tqdm import tqdm
from neo4j import GraphDatabase
import networkx as nx

# connext to neo4j instance
URI = "YOUR_URI"
AUTH = ("USERNAME", "PASSWORD")
driver = GraphDatabase.driver(URI, auth=AUTH)

# create graph
G = nx.DiGraph()

def make_label(properties):
    """
    Convert node properties into readable label text.
    Priority:
        1. text (shortened)
        2. fileName + page_number
        3. type
        4. element ID
    """
    # if node has "text"
    if "text" in properties and properties["text"]:
        t = properties["text"].strip().replace("\n", " ")
        return t[:60] + "..." if len(t) > 60 else t

    # if node has filename + page
    if "fileName" in properties and "page_number" in properties:
        return f"{properties['fileName']} (page {properties['page_number']})"

    # if node has a type
    if "type" in properties:
        return properties["type"]

    # if everything else fails, element id
    return properties.get("id", properties.get("element_id", "Unknown Node"))

# query and generate graph
node_ids = []
edges = []

with driver.session() as session:
    results = session.run("MATCH (n)-[r]->(m) RETURN n, r, m")

    for record in results:
        n = record["n"]
        m = record["m"]

        # unique node IDs
        n_id = n.element_id
        m_id = m.element_id

        # give the nodes labels
        G.add_node(n_id, label=make_label(n))
        G.add_node(m_id, label=make_label(m))

        # record edges
        edges.append((n_id, m_id))

# add edges to graph
G.add_edges_from(edges)

# get the most important nodes in the knowledge graph
centrality = nx.degree_centrality(G)
sorted_nodes = sorted(centrality.items(), key=lambda x: -x[1])
top_k = 10

global_kg_context = [
    G.nodes[node]["label"] for node, _ in sorted_nodes[:top_k]
]

# load gemma
model_id = "google/gemma-3-4b-it"
torch_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch_dtype,
    device_map="auto"
)
model.eval()
model.set_attn_implementation("eager")
model.config.use_cache = False

# load mines.csv
df = pd.read_csv("mines.csv")

# uncomment for testing
# df = df.head(2)
dataset = Dataset.from_pandas(df)

# prompt with knowledge graph context
def build_prompt_with_kg(batch, global_kg_context, max_items=5):
    prompts = []
    kg_text = " | ".join(global_kg_context[:max_items])

    for context, target in zip(batch["Context"], batch["Target"]):
        prompt = (
            f"You are a sensor expert. Given the following situation:\n"
            f"Context: {context}\n"
            f"KG context: {kg_text}\n"
            f"Target: Land mine {target}\n\n"
            f"Recommend the most appropriate type of sensor..."
        )
        prompts.append(prompt)

    return {"prompt": prompts}


dataset = dataset.map(
    lambda batch: build_prompt_with_kg(batch, global_kg_context),
    batched=True
)


# output location
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
base_dir = f"/content/drive/MyDrive/CMDACapstone_{timestamp}"
attn_dir = os.path.join(base_dir, "attentions")
os.makedirs(attn_dir, exist_ok=True)

# Run inference and save attention weights
@torch.inference_mode()
def generate_with_attentions(batch, indices):
    responses, attn_files = [], []

    for prompt, idx in zip(batch["prompt"], indices):
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, padding=True).to(model.device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.7,
            return_dict_in_generate=True,
            output_attentions=True
        )

        full_response = tokenizer.decode(outputs.sequences[0], skip_special_tokens=True).strip()
        responses.append(full_response)

        # Save attention weights from final generation step
        if outputs.attentions is not None:
            final_step = outputs.attentions[-1]  # last token generation
            mean_layers = []
            for layer_attn in final_step:
                if isinstance(layer_attn, torch.Tensor):
                    mean_layers.append(layer_attn.mean(dim=(0, 1)))  # mean over batch & heads

            mean_layers = [layer.cpu().tolist() for layer in mean_layers]
            attn_path = os.path.join(attn_dir, f"attn_{idx:04d}.json")
            with open(attn_path, "w") as f:
                json.dump({"mean_attention_final_step": mean_layers}, f)
            attn_files.append(attn_path)
        else:
            attn_files.append(None)

    return {"full_response": responses, "attention_file": attn_files}

dataset = dataset.map(
    generate_with_attentions,
    with_indices=True,
    batched=True,
    batch_size=8
)

# csv file
df_out = dataset.to_pandas()[["full_response", "attention_file"]]
csv_path = os.path.join(base_dir, "sensor_recommendations.csv")
df_out.to_csv(csv_path, index=False)

# zip file
zip_path = os.path.join(base_dir, "attentions_full.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    for f in tqdm(os.listdir(attn_dir), total=len(os.listdir(attn_dir)), leave=False):
        full_path = os.path.join(attn_dir, f)
        zipf.write(full_path, arcname=f)

print(f"CSV Saved: {csv_path}")
print(f"Attention JSONs: {attn_dir}")
print(f"Zipped attentions: {zip_path}")
print(df_out.head(3))


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Parameter 'function'=<function generate_with_attentions at 0x7f39981b99e0> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


KeyboardInterrupt: 